# 05 — Compare observed spacings with 04 surrogates

Comparison only: no acquisition, unfolding, or surrogate generation.

In [ ]:
from pathlib import Path
import numpy as np

UNFOLDED_FILE = Path("data/derived/unfolded_spacings.float64")
assert UNFOLDED_FILE.exists(), f"Missing {UNFOLDED_FILE}. Run 03_unfold first."
unfolded = np.fromfile(UNFOLDED_FILE, dtype=np.float64)
assert unfolded.ndim == 1
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)
print("observed/unfolded:", len(unfolded))
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))

In [ ]:
required = {
    "shuffle": "surrogate_shuffle",
    "iid": "surrogate_iid",
    "uniform": "surrogate_uniform",
}
missing = [name for name, var in required.items() if var not in globals()]
assert not missing, (
    "Missing 04_surrogates result(s): " + ", ".join(missing)
    + ". Run 04_surrogates in this Colab runtime before 05_compare."
)
datasets = {
    "observed": unfolded,
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}
for name, values in datasets.items():
    assert values.ndim == 1
    assert len(values) == len(unfolded)
    assert np.all(np.isfinite(values))
    assert np.all(values >= 0)
print("datasets:", ", ".join(datasets))
print("N:", len(unfolded))

In [ ]:
print("=== distribution summary ===")
for name, values in datasets.items():
    print(f"{name:>8}: mean={np.mean(values):.8f}  std={np.std(values):.8f}  min={np.min(values):.8f}  max={np.max(values):.8f}")

In [ ]:
PERCENTILES = [1, 5, 25, 50, 75, 95, 99]
print("=== quantiles ===")
for name, values in datasets.items():
    q = np.percentile(values, PERCENTILES)
    print(name, {p: float(v) for p, v in zip(PERCENTILES, q)})

In [ ]:
def lag1_correlation(values):
    return np.corrcoef(values[:-1], values[1:])[0, 1]

print("=== lag-1 correlation ===")
for name, values in datasets.items():
    print(f"{name:>8}: {lag1_correlation(values):+.8f}")

In [ ]:
BLOCK_SIZE = 1000
def block_means(values, block_size=BLOCK_SIZE):
    n = len(values) // block_size
    return np.asarray([np.mean(values[i * block_size:(i + 1) * block_size]) for i in range(n)])

print("=== block-mean variation ===")
for name, values in datasets.items():
    means = block_means(values)
    print(f"{name:>8}: blocks={len(means)}  min={means.min():.8f}  max={means.max():.8f}  std={means.std():.8f}")

In [ ]:
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))
assert np.isclose(np.mean(surrogate_shuffle), np.mean(unfolded))
assert np.isclose(np.std(surrogate_shuffle), np.std(unfolded))
print("\nALL COMPARISON INPUTS AND BASIC INVARIANTS PASSED")